<a href="https://colab.research.google.com/github/by-aseel/colab-practice/blob/main/practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Hello! This is my first GitHub practice.")

Hello! This is my first GitHub practice.


yes that is it

In [13]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import requests
import zipfile

from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image

# ============================================================
# 0. Download and Extract the Dataset (Fixed with User-Agent)
# ============================================================

dataset_url = 'https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip'
zip_path = 'cats_and_dogs_filtered.zip'
extract_dir = os.path.expanduser('~/.keras/datasets')

# Make sure the target cache directory exists
os.makedirs(extract_dir, exist_ok=True)

# Download using requests with a real User-Agent header to avoid 403 Forbidden
if not os.path.exists(os.path.join(extract_dir, 'cats_and_dogs_filtered')):
    print("Downloading dataset with custom User-Agent headers...")
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'
    }
    response = requests.get(dataset_url, headers=headers, stream=True)
    response.raise_for_status()

    local_zip_file = os.path.join(extract_dir, zip_path)
    with open(local_zip_file, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

    print("Extracting dataset...")
    with zipfile.ZipFile(local_zip_file, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    # Clean up zip file after extraction
    os.remove(local_zip_file)
    print("Dataset downloaded and extracted successfully!")
else:
    print("Dataset already exists in cache, skipping download.")

base_dir = os.path.join(extract_dir, 'cats_and_dogs_filtered')
TRAIN_DIR = os.path.join(base_dir, 'train')
VALIDATION_DIR = os.path.join(base_dir, 'validation')

# ============================================================
# 1. Parameters
# ============================================================

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10


# ============================================================
# 2. Load the dataset
# ============================================================

train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    VALIDATION_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

# Check class names
class_names = train_dataset.class_names

print("Classes:", class_names)


# ============================================================
# 3. Improve data loading performance
# ============================================================

AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)


# ============================================================
# 4. Data augmentation
# ============================================================

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])


# ============================================================
# 5. Build CNN model
# ============================================================

model = models.Sequential([

    # Data augmentation
    data_augmentation,

    # Normalize pixel values from [0,255] to [0,1]
    layers.Rescaling(1./255),

    # First convolution block
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D(),

    # Second convolution block
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D(),

    # Third convolution block
    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D(),

    # Convert feature maps into a vector
    layers.Flatten(),

    # Fully connected layer
    layers.Dense(128, activation="relu"),

    # Prevent overfitting
    layers.Dropout(0.5),

    # Binary classification
    layers.Dense(1, activation="sigmoid")
])


# ============================================================
# 6. Compile the model
# ============================================================

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Display model architecture
model.summary()


# ============================================================
# 7. Train the model
# ============================================================

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS
)


# ============================================================
# 8. Evaluate the model
# ============================================================

loss, accuracy = model.evaluate(validation_dataset)

print("\nValidation Accuracy:", accuracy)
print("Validation Loss:", loss)


# ============================================================
# 9. Plot training results
# ============================================================

plt.figure(figsize=(12, 5))

# Accuracy
plt.subplot(1, 2, 1)

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()


# Loss
plt.subplot(1, 2, 2)

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()


# ============================================================
# 10. Save the model
# ============================================================

model.save("cat_dog_classifier.keras")

print("\nModel saved as cat_dog_classifier.keras")


# ============================================================
# 11. Predict a new image
# ============================================================

def predict_image(img_path):
    if not os.path.exists(img_path):
        print(f"Test image '{img_path}' not found. Skipping prediction.")
        return

    # Load image
    img = image.load_img(
        img_path,
        target_size=IMG_SIZE
    )

    # Convert image to NumPy array
    img_array = image.img_to_array(img)

    # Add batch dimension
    img_array = np.expand_dims(img_array, axis=0)

    # Make prediction
    prediction = model.predict(img_array, verbose=0)[0][0]

    # Display image
    plt.imshow(img)
    plt.axis("off")

    # Since we use binary classification:
    # 0 = cat
    # 1 = dog
    if prediction < 0.5:
        predicted_class = "Cat"
        confidence = (1 - prediction) * 100
    else:
        predicted_class = "Dog"
        confidence = prediction * 100

    plt.title(
        f"Prediction: {predicted_class}\n"
        f"Confidence: {confidence:.2f}%"
    )

    plt.show()

    print(f"Prediction: {predicted_class}")
    print(f"Confidence: {confidence:.2f}%")


# ============================================================
# 12. Test a new image
# ============================================================

# Note: You can replace this path with a real local image to run predictions
predict_image("test_image.jpg")

Exception: URL fetch failure on https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip: 403 -- Forbidden